In [ ]:
# from google.colab import drive
# drive.mount("/content/drive/")

In [ ]:
# Important: cd into mace directory
# %cd mace

In [ ]:
# # --- Optional: Download and Install Miniconda ---
# !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
# !bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local
# !conda init bash
# !conda config --set auto_update_conda false
# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
# --- Create the Conda environment with all necessary compilers and libraries ---
!conda create -n mace --yes python=3.10 \
  "pytorch==2.0.1" \
  "torchvision==0.15.2" \
  "pytorch-cuda=11.7" \
  "cuda-toolkit=11.7" \
  "gcc_linux-64=9.4.0" \
  "gxx_linux-64=9.4.0" \
  "cmake" \
  -c pytorch -c nvidia -c conda-forge

In [ ]:
!conda install -n mace -c conda-forge gcc=9.4.0 gxx=9.4.0 -y

In [ ]:
# Install CUDA 11.7 systemwide
%%bash
sudo apt-get update -y
sudo apt-get install -y cuda-toolkit-11-7

Should show gcc & g++ 9.4.0

In [ ]:
!conda run -n mace g++ --version
!conda run -n mace gcc --version

CUDA should show 11.7 for both torch and nvcc

In [ ]:
!conda run -n mace python -c "import torch; print(torch.version.cuda, torch.__version__)"
!conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && nvcc --version"

In [ ]:
!conda run -n mace pip install numpy==1.26.4 --force-reinstall

In [ ]:
!conda run -n mace bash -c "pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117"

In [ ]:
# Install grounded-SAM
!conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && pip install -vvv --no-build-isolation Grounded-Segment-Anything/GroundingDINO/"

Install the other libraries needed for MACE

In [ ]:
!conda run -n mace bash setup.sh

In [ ]:
!conda run -n mace pip install diffusers==0.22.0 transformers==4.46.2 huggingface_hub==0.25.2
!conda run -n mace pip install accelerate openai omegaconf opencv-python

In [ ]:
!wget -O Grounded-Segment-Anything/groundingdino_swint_ogc.pth \
https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

!wget -O Grounded-Segment-Anything/sam_hq_vit_h.pth \
https://huggingface.co/lkeab/hq-sam/resolve/main/sam_hq_vit_h.pth

In [ ]:
!conda run -n mace pip install numpy==1.26.4 --force-reinstall

Prepare the data

In [ ]:
%%time
!unset MPLBACKEND && conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && CUDA_VISIBLE_DEVICES=0 python data_preparation.py configs/object/erase_bluetick.yaml"

In [ ]:
%%time
!unset MPLBACKEND && conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && CUDA_VISIBLE_DEVICES=0 python data_preparation.py configs/object/erase_chesapeake.yaml"

Train the model

In [ ]:
%%time
!conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && CUDA_VISIBLE_DEVICES=0 python training.py configs/object/erase_chesapeake.yaml"

# Generate images

In [ ]:
%%time
!conda run -n mace bash -c "export CUDA_HOME=/usr/local/cuda-11.7 && export PATH=\$CUDA_HOME/bin:\$PATH && export LD_LIBRARY_PATH=\$CUDA_HOME/lib64:\$LD_LIBRARY_PATH && CUDA_VISIBLE_DEVICES=0 accelerate launch --num_processes=1 --main_process_port 31372 src/sample_images_from_csv.py --prompts_path ./prompts_csv/mace_generate_chesapeake.csv --save_path '/content/drive/My Drive/thesis/fgst-de/MACE/chesapeake' --model_name saved_model/LoRA_fusion_model --step 1"